# Microsoft Presidio: An Introduction to PII Detection and Anonymization

This notebook provides a detailed overview of Microsoft Presidio, a service for analyzing text and identifying personally identifiable information (PII). It also demonstrates how to anonymize the detected PII using various techniques.

## What is Microsoft Presidio?

Microsoft Presidio is an open-source tool that helps organizations ensure data privacy by automatically detecting, classifying, and anonymizing sensitive information like names, credit card numbers, email addresses, and more within unstructured text. It consists of two main components:

1.  **Presidio Analyzer**: Detects and identifies PII entities in text.
2.  **Presidio Anonymizer**: Anonymizes the detected PII entities using various methods (e.g., replace, mask, redact, hash, encrypt).

## Installation

First, let's install the necessary Presidio libraries. We'll need `presidio_analyzer` for PII detection and `presidio_anonymizer` for anonymization.

In [ ]:
!pip install presidio_analyzer presidio_anonymizer faker -q

## Basic PII Detection with Presidio Analyzer

Let's start by using the Presidio Analyzer to detect PII in a sample text. The analyzer returns a list of `RecognizerResult` objects, each indicating the detected entity, its location, and the score.

In [ ]:
import logging
logging.getLogger("presidio-analyzer").setLevel(logging.ERROR)

In [ ]:
from presidio_analyzer import AnalyzerEngine, RecognizerResult

# Initialize the AnalyzerEngine
analyzer = AnalyzerEngine()

# Sample text containing PII
text = "My name is John Doe, and my phone number is (555) 123-4567. My email is john.doe@example.com. I live at 123 Main Street, New York, NY."

print(f"Original Text:\n{text}\n")

# Analyze the text for PII
results = analyzer.analyze(text=text, language='en')

# Print the detected PII entities
print("Detected PII entities:")
for result in results:
    print(f"- Entity: {result.entity_type}, Text: {text[result.start:result.end]}, Score: {result.score:.2f}, Position: ({result.start}, {result.end})")


Original Text:
My name is John Doe, and my phone number is (555) 123-4567. My email is john.doe@example.com. I live at 123 Main Street, New York, NY.

Detected PII entities:
- Entity: EMAIL_ADDRESS, Text: john.doe@example.com, Score: 1.00, Position: (72, 92)
- Entity: PERSON, Text: John Doe, Score: 0.85, Position: (11, 19)
- Entity: LOCATION, Text: 123 Main Street, Score: 0.85, Position: (104, 119)
- Entity: LOCATION, Text: New York, Score: 0.85, Position: (121, 129)
- Entity: LOCATION, Text: NY, Score: 0.85, Position: (131, 133)
- Entity: PHONE_NUMBER, Text: (555) 123-4567, Score: 0.75, Position: (44, 58)
- Entity: URL, Text: john.do, Score: 0.50, Position: (72, 79)
- Entity: URL, Text: example.com, Score: 0.50, Position: (81, 92)


## Anonymization with Presidio Anonymizer

Once PII is detected, the Presidio Anonymizer can replace or transform this sensitive information. We'll demonstrate a simple replacement strategy.

In [ ]:
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.operators import OperatorType

# Initialize the AnonymizerEngine
anonymizer = AnonymizerEngine()

# Anonymize the text using the detected results
anonymized_result = anonymizer.anonymize(text=text, analyzer_results=results)

print(f"Anonymized Text (default replacement):\n{anonymized_result.text}\n")


Anonymized Text (default replacement):
My name is <PERSON>, and my phone number is <PHONE_NUMBER>. My email is <EMAIL_ADDRESS>. I live at <LOCATION>, <LOCATION>, <LOCATION>.



### Custom Anonymization Operators

Presidio allows for custom anonymization strategies using different operators. For example, we can replace names with a generic placeholder, phone numbers with a mask, and emails with a randomly generated email using `Faker`.

In [ ]:
from presidio_anonymizer.operators import OperatorType
from presidio_anonymizer.entities import OperatorConfig
from faker import Faker

fake = Faker()

# Define custom anonymization operators
anonymization_operators = {
    "DEFAULT": OperatorConfig("replace", {"new_value": "<ANONYMIZED>"}),
    "PERSON": OperatorConfig("replace", {"new_value": "<NAME>"}),
    "PHONE_NUMBER": OperatorConfig("mask", {"masking_char": "*", "chars_to_mask": 10, "from_end": False}), # Changed to use chars_to_mask
    "EMAIL_ADDRESS": OperatorConfig("custom", {"lambda": lambda x: fake.email()})
}

# Re-anonymize with custom operators
custom_anonymized_result = anonymizer.anonymize(
    text=text,
    analyzer_results=results,
    operators=anonymization_operators
)

print(f"Anonymized Text (custom operators):\n{custom_anonymized_result.text}\n")

Anonymized Text (custom operators):
My name is <NAME>, and my phone number is **********4567. My email is murrayjesse@example.org. I live at <ANONYMIZED>, <ANONYMIZED>, <ANONYMIZED>.



---